In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [2]:
import pandas as pd
import nltk
import re
import matplotlib.pyplot as plt
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split
import spacy
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import KFold

In [3]:
nlp = spacy.load("en_core_web_sm")

news_df = pd.read_csv('/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv')
df = news_df
news_df.head()

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score
0,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85
1,2023-01-12 10:34:00,EURCHF,Positive,EURCHF Trend higher may remain in place – ING,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-trend-hi...,FX Street,EUR/CHF yesterday broke above 1.00. Economists...,Positive,0.51
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37
3,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64
4,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83


This section of codes belong to "moreCleaned_Emotion_Detection,_HLT_project.ipynb"

In [4]:
# Handling missing values
news_df.fillna('', inplace=True)

# Removing duplicate rows
news_df = news_df.drop_duplicates()

# Converting to lowercase
news_df['title'] = news_df['title'].str.lower()
news_df['text'] = news_df['text'].str.lower()

# Removing punctuation
news_df['title'] = news_df['title'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
news_df['text'] = news_df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# Removing titles with less than 3 word, texts with less than 50 words
news_df = news_df[news_df['title'].apply(lambda x: len(x.split()) >= 3)]
news_df = news_df[news_df['text'].apply(lambda x: len(x.split()) >= 50)]

In [5]:
def tokenize_and_lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

news_df['titleTokens'] = news_df['title'].apply(tokenize_and_lemmatize)
news_df['textTokens'] = news_df['text'].apply(tokenize_and_lemmatize)

In [6]:
def load_lexicon(filepath):
    lexicon = {}
    with open(filepath, 'r') as file:
        for line in file:
            word, valence, arousal, dominance = line.strip().split('\t')
            lexicon[word.lower()] = {'valence': float(valence), 'arousal': float(arousal), 'dominance': float(dominance)}
    return lexicon

lexicon = load_lexicon('/content/drive/MyDrive/HLT project/Project files/NRC-VAD-Lexicon.txt')

In [9]:
list(lexicon)[:5]

['aaaaaaah', 'aaaah', 'aardvark', 'aback', 'abacus']

Create a bag of words dictionary for each class and in total. For creating the file, we will go through each "document" and add  all of our data to the bag of words dictionary. We will ignore all of the words which will have only special characters (dot, comma, paranthesis etc.), and also numbers due to not actually helping a lot in terms of sentiment analysis.

The article of the dataset says that the data is split in 6 key pair categories which are relevant: AUDUSD, EURCHF, EURUSD, GBPUSD, and USDJPY. We will apply naive bayes with this variable in addition as a prior knowledge, with the assumption that it is independent with the data and classes. "t" will be the pair class.

$$P(c|d,t) ∝ P(d,t|c)*P(c) = P(d|c)*P(t|c)*P(c) $$



In [10]:
def create_bag_of_words_occurences_ticker(df, label):
  bag_dict = {
      "Positive": {},
      "Positive_count": 0,
      "Positive_count_frequency": 0,
      "Neutral": {},
      "Neutral_count": 0,
      "Neutral_count_frequency": 0,
      "Negative": {},
      "Negative_count": 0,
      "Negative_count_frequency": 0,
      "total": {},
      "total_count_frequency": 0,
      "total_count": len(df)
  }

  regex_filter = "[A-Za-z]+"

  for i in df.index:

    bag_dict[df["true_sentiment"][i] + '_count'] += 1
    doc_tokenized = [el.lower() for el in df[label + 'Tokens'][i] if re.search(regex_filter, el)]
    occurence_words = {}
    ticker = df["ticker"][i]
    if ticker not in bag_dict[df["true_sentiment"][i]]:
      bag_dict[df["true_sentiment"][i]][ticker] = 1
    else:
      bag_dict[df["true_sentiment"][i]][ticker] += 1

    if ticker not in bag_dict["total"]:
      bag_dict["total"][ticker] = 1
    else:
      bag_dict["total"][ticker] += 1

    for el in doc_tokenized:
      if el not in occurence_words:
        occurence_words[el] = 0
        if el not in bag_dict[df["true_sentiment"][i]]:
          bag_dict[df["true_sentiment"][i]][el] = 1
        else:
          bag_dict[df["true_sentiment"][i]][el] += 1
        if el not in bag_dict["total"]:
          bag_dict["total"][el] = 1
        else:
          bag_dict["total"][el] += 1
        bag_dict[df["true_sentiment"][i] + '_count_frequency'] += 1
        bag_dict["total_count_frequency"] += 1

  return bag_dict

def get_predictions_ticker(doc_dict, df):
  prediction = []
  regex_filter = "[A-Za-z]+"
  classes = ["Positive", "Neutral", "Negative"]
  for i in df.index:

      doc_tokenized = [el.lower() for el in df[label + 'Tokens'][i] if re.search(regex_filter, el)]

      results = {}
      V_len = len(doc_dict["total"].keys())
      for cls in classes:
        results[cls] = np.log(doc_dict[cls + "_count"] / doc_dict["total_count"])
        results[cls] += np.log(doc_dict[cls][df["ticker"][i]] / doc_dict["total"][df["ticker"][i]])
        cls_freq = doc_dict[cls + "_count_frequency"]
        for el in doc_tokenized:
          # we apply LaPlace (adding + 1 to each word)
          word_count = doc_dict[cls][el] + 1 if el in doc_dict[cls] else 1
          results[cls] += np.log(word_count / (cls_freq + V_len))

      prediction.append(max(results, key=results.get))

  return prediction

In [11]:
df_train, df_test = train_test_split(news_df, test_size=0.2)
label = "text"
text_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker(text_dict, df_test)
print("Occurences metrics for the text:")
print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

label = "title"
print("Occurences metrics for the title:")
text_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker(text_dict, df_test)

print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Occurences metrics for the text:
              precision    recall  f1-score   support

    Positive       0.51      0.50      0.50       157
     Neutral       0.49      0.51      0.50       159
    Negative       0.45      0.45      0.45       139

    accuracy                           0.49       455
   macro avg       0.48      0.48      0.48       455
weighted avg       0.49      0.49      0.49       455

Occurences metrics for the title:
              precision    recall  f1-score   support

    Positive       0.60      0.72      0.65       157
     Neutral       0.61      0.50      0.55       159
    Negative       0.62      0.61      0.62       139

    accuracy                           0.61       455
   macro avg       0.61      0.61      0.61       455
weighted avg       0.61      0.61      0.61       455



Kfold for the last one

In [12]:
kf = KFold(5)

for i, (train_index, test_index) in enumerate(kf.split(news_df)):
  df_train = news_df.iloc[train_index]
  df_test = news_df.iloc[test_index]
  print("Kfold {}".format(i))
  print("For text")
  label = "text"
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker(text_dict, df_test)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))
  print("For title")
  label = "title"
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker(text_dict, df_test)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Kfold 0
For text
              precision    recall  f1-score   support

    Positive       0.39      0.43      0.41       134
     Neutral       0.41      0.50      0.45       147
    Negative       0.47      0.34      0.40       174

    accuracy                           0.42       455
   macro avg       0.42      0.43      0.42       455
weighted avg       0.43      0.42      0.42       455

For title
              precision    recall  f1-score   support

    Positive       0.57      0.66      0.61       134
     Neutral       0.54      0.49      0.51       147
    Negative       0.64      0.62      0.63       174

    accuracy                           0.59       455
   macro avg       0.59      0.59      0.59       455
weighted avg       0.59      0.59      0.59       455

Kfold 1
For text
              precision    recall  f1-score   support

    Positive       0.36      0.50      0.41       137
     Neutral       0.47      0.51      0.49       155
    Negative       0.53      0.

In [20]:
def get_predictions_ticker_w_lexicon(doc_dict, df, lexicon, attrs = []):
  prediction = []
  regex_filter = "[A-Za-z]+"
  classes = ["Positive", "Neutral", "Negative"]
  for i in df.index:

      doc_tokenized = [el.lower() for el in df[label + 'Tokens'][i] if re.search(regex_filter, el)]

      results = {}
      V_len = len(doc_dict["total"].keys())
      for cls in classes:
        results[cls] = np.log(doc_dict[cls + "_count"] / doc_dict["total_count"])
        results[cls] += np.log(doc_dict[cls][df["ticker"][i]] / doc_dict["total"][df["ticker"][i]])
        cls_freq = doc_dict[cls + "_count_frequency"]
        for el in doc_tokenized:
          # we apply LaPlace (adding + 1 to each word)
          if el in lexicon.keys():
            word_count = doc_dict[cls][el] + 1 if el in doc_dict[cls] else 1
            results[cls] += np.log(word_count / (cls_freq + V_len))

      prediction.append(max(results, key=results.get))

  return prediction

In [21]:
df_train, df_test = train_test_split(news_df, test_size=0.2)
label = "text"
text_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker_w_lexicon(text_dict, df_test, lexicon)
print("Occurences metrics for the text:")
print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

label = "title"
print("Occurences metrics for the title:")
text_dict = create_bag_of_words_occurences_ticker(df_train, label)

predictions = get_predictions_ticker_w_lexicon(text_dict, df_test, lexicon)

print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Occurences metrics for the text:
              precision    recall  f1-score   support

    Positive       0.51      0.47      0.49       154
     Neutral       0.56      0.53      0.55       163
    Negative       0.43      0.49      0.46       138

    accuracy                           0.50       455
   macro avg       0.50      0.50      0.50       455
weighted avg       0.50      0.50      0.50       455

Occurences metrics for the title:
              precision    recall  f1-score   support

    Positive       0.60      0.64      0.62       154
     Neutral       0.63      0.56      0.59       163
    Negative       0.60      0.62      0.61       138

    accuracy                           0.61       455
   macro avg       0.61      0.61      0.61       455
weighted avg       0.61      0.61      0.61       455



In [23]:
kf = KFold(5)

for i, (train_index, test_index) in enumerate(kf.split(news_df)):
  df_train = news_df.iloc[train_index]
  df_test = news_df.iloc[test_index]
  print("Kfold {}".format(i))
  print("For text")
  label = "text"
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker_w_lexicon(text_dict, df_test, lexicon)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))
  print("For title")
  label = "title"
  text_dict = create_bag_of_words_occurences_ticker(df_train, label)
  predictions = get_predictions_ticker_w_lexicon(text_dict, df_test, lexicon)
  print(classification_report(df_test["true_sentiment"], predictions, labels=["Positive", "Neutral", "Negative"]))

Kfold 0
For text
              precision    recall  f1-score   support

    Positive       0.41      0.38      0.40       134
     Neutral       0.38      0.51      0.43       147
    Negative       0.47      0.36      0.41       174

    accuracy                           0.41       455
   macro avg       0.42      0.42      0.41       455
weighted avg       0.42      0.41      0.41       455

For title
              precision    recall  f1-score   support

    Positive       0.55      0.61      0.58       134
     Neutral       0.53      0.54      0.53       147
    Negative       0.65      0.58      0.61       174

    accuracy                           0.58       455
   macro avg       0.57      0.58      0.57       455
weighted avg       0.58      0.58      0.58       455

Kfold 1
For text
              precision    recall  f1-score   support

    Positive       0.38      0.47      0.42       137
     Neutral       0.45      0.57      0.50       155
    Negative       0.60      0.

From an article ("Sentiment Analysis Using Naive Bayes with Lexicon-Based Feature on
TikTok Application"), we find a formula which will help us calculate the naive bayes classifier predictor:

$$P(c|d,t) ∝ P(d|c)*P(t|c) + lexicon\_score * P(c) $$